In [1]:
import pandas as pd
df=pd.read_csv('customer_shopping_behavior.csv')
df.shape


(3900, 18)

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [3]:
df.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [4]:
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [5]:
# Check mean, median and missing values of Review Rating by Category

category_stats = (
    df.groupby('Category')['Review Rating']
      .agg(
          mean='mean',
          median='median',
          missing=lambda x: x.isnull().sum()
      )
      .reset_index()
)

print(category_stats)

      Category      mean  median  missing
0  Accessories  3.769976     3.8       11
1     Clothing  3.721537     3.7       19
2     Footwear  3.793771     3.8        5
3    Outerwear  3.745652     3.8        2


Here we can see mean and median of each category has not that much difference so here we can also use overall rating mean and meadian ,
But more reliable is to use category wise imputation 

In [6]:
df['Review Rating']= df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [7]:
df.columns=df.columns.str.strip().str.replace(' ', '_').str.lower()

In [8]:
df=df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})
print(df.columns)

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')


In [9]:
labels=['Young Adult','Adult','Middle_aged','Senior']
df['age_group']=pd.cut(df['age'],bins=[16,30,50,65,100],labels=labels)

In [10]:
df['frequency_of_purchases'].unique()

array(['Fortnightly', 'Weekly', 'Annually', 'Quarterly', 'Bi-Weekly',
       'Monthly', 'Every 3 Months'], dtype=object)

In [11]:
# purchase_frequency days 
frequuency_mapping = {'Fortnightly': 14, 'Monthly': 30, 'Weekly': 7, 'Daily': 1, 'Quarterly': 90, 'Annually': 365, 'Bi-Weekly': 14,'Every 3 Months': 90} 
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequuency_mapping)

In [12]:
# check and remove the redunadant columns
df[['discount_applied', 'promo_code_used']].head()

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes


In [13]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [14]:
df.drop(columns=['promo_code_used'], inplace=True)

print(df.columns)

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')


In [15]:
%pip install psycopg2-binary sqlalchemy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [16]:
%pip install sqlalchemy pyodbc

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
from sqlalchemy import create_engine

In [18]:
import pyodbc

conn = pyodbc.connect(
    """
    DRIVER={ODBC Driver 17 for SQL Server};
    SERVER=localhost;
    Trusted_Connection=yes;
    """
)

print("Connected Successfully")

Connected Successfully


In [19]:
from sqlalchemy import create_engine
import urllib

database = "customer_behaviour_database"

params = urllib.parse.quote_plus(
    f"""
    DRIVER={{ODBC Driver 17 for SQL Server}};
    SERVER=localhost;
    DATABASE={database};
    Trusted_Connection=yes;
    """
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

df.to_sql(
    "customer_shopping_behavior",
    con=engine,
    if_exists="replace",
    index=False
)

print("Data uploaded successfully")

c:\ProgramData\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Data uploaded successfully
